# Mason & Monsignori Fossi (1994) — VUV Spectroscopic Diagnostics / VUV 분광 진단법 구현

**Paper**: H. E. Mason & B. C. Monsignori Fossi, *"Spectroscopic diagnostics in the VUV for solar and stellar plasmas"*, **A&A Review 6, 123–179 (1994)**.

## Goal / 목표

This notebook implements the *core diagnostic techniques* taught in **§3** of the review using analytic two-level / multi-level approximations (since ChiantiPy is not installed in this environment).

이 노트북은 본 종설 §3에서 제시된 핵심 진단 기법을 ChiantiPy 없이 해석적 2-레벨/다레벨 근사로 재현한다.

We cover five canonical diagnostics — density-sensitive line ratios (Eq. 23–28), temperature-sensitive line ratios (Eq. 29), forward emission-measure synthesis, Pottasch isothermal EM inversion (Eq. 31), and FIP-effect visualisation (Fig. 4) — and finish with a Summary table linking the techniques to modern descendants (#59 Del Zanna & Mason 2018, #61 Abbo+ 2025, #63/#64 CHIANTI).

본 노트북은 5가지 정전 진단법(밀도 민감 비율, 온도 민감 비율, 순방향 EM 합성, Pottasch 등온 EM 역산, FIP 효과 시각화)을 다루고 마지막으로 현대 후속 논문들과의 연결을 정리한다.

In [ ]:
"""Common imports and physical constants for Mason 1994 diagnostics.

All numerical constants are CGS / atomic units consistent with the review.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Boltzmann constant in eV/K -- required to evaluate exp(-DeltaE / kT_e).
K_B_EV = 8.617333262e-5

# Convenience random-number generator for reproducible illustrative noise.
RNG = np.random.default_rng(seed=1994)

print(f"NumPy {np.__version__} ready, k_B = {K_B_EV:.4e} eV/K")

## Part 1: Density-sensitive line ratio (Eq. 23–28) / 밀도 민감 비율

### Theory / 이론

For an ion with a metastable level $m$ and the ground level $g$, the ratio of a forbidden / intersystem line $I_{m,g}$ to a reference allowed line $I_\mathrm{ref}$ behaves as (Mason 1994, Eq. 23–28):

$$\frac{N_m}{N_g} \;=\; \frac{N_e\, C^e_{g,m}}{A_{m,g} + N_e\, C^e_{m,g}}.$$

메타스테이블 레벨 $m$과 바닥 레벨 $g$를 가진 이온의 경우, 금지/인터컴비네이션 라인 $I_{m,g}$와 알로우드 기준선 $I_\mathrm{ref}$의 비율은 위와 같이 표현된다.

* **Low-density limit** $N_e \to 0$: $N_e C^e_{m,g} \ll A_{m,g}$, so $I_{m,g} \propto N_e^2$ (Eq. 24).
* **High-density limit** $N_e \to \infty$: collisions dominate, populations Boltzmannise, and $I_{m,g} \propto N_e$ (Eq. 26).
* The **transition density** $N_e^\star \equiv A_{m,g} / C^e_{m,g}$ marks the cross-over.

저밀도 한계에서는 $I_{m,g} \propto N_e^2$, 고밀도 한계에서는 $I_{m,g} \propto N_e$이며, 전이 밀도는 $N_e^\star = A_{m,g}/C^e_{m,g}$이다.

We illustrate with two transition-region ratios:

* **O IV [1407.39 / 1404.81]** — almost $T_e$-insensitive, sensitive between $10^{10}$–$10^{12}$ cm⁻³.
* **C III [1909 / 977]** — sensitive between $10^{8}$–$10^{11}$ cm⁻³.

Generic atomic data $A_{m,g} = 10\;\mathrm{s^{-1}}$ and $C^e_{g,m} = 10^{-9}\;\mathrm{cm^3\,s^{-1}}$ are used for illustration.

In [ ]:
@dataclass(frozen=True)
class TwoLevelIon:
    """Two-level metastable atom for density-sensitive line ratios.

    Implements the simplified statistical equilibrium of Mason 1994 Eq. 23.

    Attributes:
        name: Display label such as ``"O IV [1407/1404]"``.
        a_mg: Spontaneous-emission coefficient A_{m,g} in s^-1 (Eq. 23).
        c_gm: Excitation collision rate C^e_{g,m} in cm^3 s^-1.
        c_mg: De-excitation collision rate C^e_{m,g} in cm^3 s^-1.
    """

    name: str
    a_mg: float
    c_gm: float
    c_mg: float

    def population_ratio(self, n_e: np.ndarray) -> np.ndarray:
        """Return N_m / N_g for an array of electron densities.

        Args:
            n_e: Electron number density in cm^-3.

        Returns:
            Population ratio N_m / N_g (Mason 1994 Eq. 23).
        """
        return (n_e * self.c_gm) / (self.a_mg + n_e * self.c_mg)

    def line_ratio(self, n_e: np.ndarray) -> np.ndarray:
        """Return I_{m,g} / I_ref normalised to the high-density saturation.

        I_{m,g} is proportional to N_m * A_{m,g} and I_ref is proportional
        to N_g * N_e (allowed line in the coronal approximation), so
        I_{m,g}/I_ref proportional to (N_m / N_g) / N_e.

        Args:
            n_e: Electron number density in cm^-3.

        Returns:
            Line-intensity ratio normalised so that the low-density value
            (where I_{m,g} proportional to N_e^2) tends to a constant.
        """
        return self.population_ratio(n_e) * self.a_mg / (n_e + 1.0)

    @property
    def transition_density(self) -> float:
        """Return N_e^star = A_{m,g} / C^e_{m,g} where the regime changes."""
        return self.a_mg / self.c_mg


# Representative transition-region ratios (illustrative atomic data).
OIV_RATIO = TwoLevelIon(
    name="O IV [1407.39 / 1404.81]",
    a_mg=10.0,
    c_gm=1.0e-9,
    c_mg=1.0e-10,
)
CIII_RATIO = TwoLevelIon(
    name="C III [1909 / 977]",
    a_mg=100.0,
    c_gm=1.0e-9,
    c_mg=1.0e-8,
)

for ion in (OIV_RATIO, CIII_RATIO):
    print(f"{ion.name:30s}  N_e* = {ion.transition_density:8.2e} cm^-3")

In [ ]:
log_ne = np.linspace(7.0, 13.0, 400)
n_e = 10.0**log_ne

fig, ax = plt.subplots()
for ion, colour in zip((OIV_RATIO, CIII_RATIO), ("tab:blue", "tab:orange")):
    ratio = ion.line_ratio(n_e)
    ratio_norm = ratio / ratio.max()
    ax.plot(log_ne, ratio_norm, lw=2, color=colour, label=ion.name)
    ax.axvline(
        np.log10(ion.transition_density),
        color=colour,
        ls="--",
        alpha=0.6,
        label=f"N_e* = {ion.transition_density:.1e} cm^-3",
    )

ax.set_xlabel(r"$\log_{10} N_e\ \mathrm{[cm^{-3}]}$")
ax.set_ylabel(r"$I_{m,g} / I_\mathrm{ref}$ (normalised)")
ax.set_title("Mason 1994 Eq. 23-28 -- density-sensitive line ratios")
ax.legend(loc="best", fontsize=10)
fig.tight_layout()
plt.show()

for ion in (OIV_RATIO, CIII_RATIO):
    print(
        f"{ion.name:30s}  log N_e* = {np.log10(ion.transition_density):.2f}"
    )

**Reading / 해석**: Each curve plateaus at high $N_e$ (collision-dominated regime, $I \propto N_e$) and rolls down at low $N_e$ (radiative regime, $I \propto N_e^2$). The dashed lines mark $N_e^\star = A_{m,g}/C^e_{m,g}$ — the diagnostic sweet spot.

각 곡선은 고밀도 영역에서 평탄해지고($I \propto N_e$, 충돌 우세) 저밀도 영역에서 급격히 감소한다($I \propto N_e^2$, 복사 우세). 점선이 진단의 sweet spot인 $N_e^\star$를 표시한다.

* O IV [1407/1404]: cross-over near $\log N_e \approx 11$ — perfect for transition-region density measurements.
* C III [1909/977]: cross-over near $\log N_e \approx 10$ — sensitive to upper-chromospheric / lower-TR conditions.

## Part 2: Temperature-sensitive line ratio (Eq. 29) / 온도 민감 비율

Two allowed lines from the *same* ion but with very different excitation energies $\Delta E_j$ and $\Delta E_k$ obey (Mason 1994, Eq. 29):

$$\frac{I_{g,j}}{I_{g,k}} \;=\; \frac{\Delta E_k\,\Upsilon_j}{\Delta E_j\,\Upsilon_k}\;\exp\!\left[\frac{\Delta E_k - \Delta E_j}{k T_e}\right].$$

두 알로우드 분광선이 동일 이온에서 서로 다른 여기 에너지를 가질 때 위 식이 성립한다 — 지수 항 때문에 $T_e$에 매우 민감하다.

We illustrate with **O V [629 / 172] Å**, which has $\Delta E_j \approx 19.7\;\mathrm{eV}$ (629 Å, 2s²–2s2p) and $\Delta E_k \approx 72\;\mathrm{eV}$ (172 Å, 2s²–2s3p). The dimensionless thermally-averaged collision strengths $\Upsilon$ are taken as generic order-unity numbers (Mason 1994 §2.2).

In [ ]:
@dataclass(frozen=True)
class AllowedLine:
    """Allowed transition with excitation energy and effective collision strength.

    Attributes:
        label: Wavelength label such as ``"O V 629 A"``.
        delta_e_ev: Excitation energy Delta E in eV (Mason 1994 Eq. 29).
        upsilon: Maxwellian-averaged collision strength Upsilon (Eq. 12).
    """

    label: str
    delta_e_ev: float
    upsilon: float


def temperature_ratio(
    line_j: AllowedLine,
    line_k: AllowedLine,
    t_e: np.ndarray,
) -> np.ndarray:
    """Return the I(j)/I(k) ratio of two allowed lines vs electron temperature.

    Implements Mason 1994 Eq. 29 in the coronal-model approximation.

    Args:
        line_j: Lower-energy allowed line.
        line_k: Higher-energy allowed line.
        t_e: Electron temperature array in K.

    Returns:
        Dimensionless line-intensity ratio I_{g,j} / I_{g,k}.
    """
    prefactor = (line_k.delta_e_ev * line_j.upsilon) / (
        line_j.delta_e_ev * line_k.upsilon
    )
    arg = (line_k.delta_e_ev - line_j.delta_e_ev) / (K_B_EV * t_e)
    return prefactor * np.exp(arg)


OV_629 = AllowedLine(label="O V 629 A (2s2 - 2s2p)", delta_e_ev=19.7, upsilon=2.5)
OV_172 = AllowedLine(label="O V 172 A (2s2 - 2s3p)", delta_e_ev=72.0, upsilon=0.4)

log_t = np.linspace(5.0, 7.0, 400)
t_e = 10.0**log_t
ratio_t = temperature_ratio(OV_629, OV_172, t_e)

fig, ax = plt.subplots()
ax.semilogy(log_t, ratio_t, lw=2, color="tab:red")
ax.axvline(np.log10(2.5e5), ls="--", color="k", alpha=0.5,
           label=r"$T_\mathrm{max}(\mathrm{O\,V}) \approx 2.5\times 10^5$ K")
ax.set_xlabel(r"$\log_{10} T_e\ \mathrm{[K]}$")
ax.set_ylabel(r"$I(629\,\mathrm{\AA}) / I(172\,\mathrm{\AA})$")
ax.set_title("Mason 1994 Eq. 29 -- O V temperature diagnostic")
ax.legend()
fig.tight_layout()
plt.show()

for log_t_query in (5.0, 5.4, 6.0, 6.5):
    val = temperature_ratio(
        OV_629, OV_172, np.array([10.0**log_t_query])
    )[0]
    print(f"log T_e = {log_t_query:.1f}  ->  I(629)/I(172) = {val:.3e}")

**Reading / 해석**: The exponential $\exp[(\Delta E_k-\Delta E_j)/kT_e]$ produces several decades of variation between $\log T_e = 5$ and $\log T_e = 7$, which is the physical basis of every line-ratio thermometer in the VUV. The dashed line shows the formation temperature of O V — the ratio is most accurate where the contribution function $G(T_e)$ peaks.

지수 항이 $\log T_e = 5$–$7$ 사이에서 수십~수백 배의 변동을 만들어내며, 이것이 모든 VUV 온도 진단의 물리적 기반이다.

## Part 3: Forward EM synthesis / 순방향 방출척도 합성

We define an idealised quiet-Sun **differential emission measure** (DEM) $\varphi(T) = N_e^2\,dV/dT$ (Mason 1994 Eq. 32) as a Gaussian in $\log T$ peaking at $\log T = 6.0$ (matching Fig. 3 of the review).

정지 태양의 미분 방출척도 $\varphi(T)$를 $\log T = 6.0$에서 피크인 가우시안으로 정의한다 (Fig. 3과 일치).

For each representative line we adopt a Gaussian-approximated contribution function $G(T)$ centred on the line's formation temperature. The predicted line intensity is then

$$I = \frac{N(\mathrm{X})}{N(\mathrm{H})}\,\int G(T)\,\varphi(T)\,dT.$$

각 분광선에 대해 형성 온도에 중심을 둔 가우시안 $G(T)$를 채택하고, 위 적분을 수치 적분으로 계산한다.

In [ ]:
@dataclass(frozen=True)
class CoronalLine:
    """Representative coronal/TR line with Gaussian-approximated G(T).

    Attributes:
        label: Display label, e.g. ``"O V 1218 A"``.
        log_t_peak: log10 of the formation temperature in K.
        sigma_log_t: Width of the Gaussian G(T) in log10 T units (typ. 0.15).
        g_peak: Peak value of G(T) in erg cm^3 s^-1 sr^-1 (illustrative).
        abundance: Element abundance N(X)/N(H) (photospheric, Asplund-like).
    """

    label: str
    log_t_peak: float
    sigma_log_t: float
    g_peak: float
    abundance: float

    def g_of_t(self, log_t: np.ndarray) -> np.ndarray:
        """Return the Gaussian-approximated contribution function G(T).

        Args:
            log_t: log10 of the electron temperature in K.

        Returns:
            G(T) sampled at each log_t (illustrative units).
        """
        z = (log_t - self.log_t_peak) / self.sigma_log_t
        return self.g_peak * np.exp(-0.5 * z * z)


def quiet_sun_dem(log_t: np.ndarray,
                  log_t_peak: float = 6.0,
                  sigma: float = 0.35,
                  amplitude: float = 1.0e22) -> np.ndarray:
    """Idealised quiet-Sun DEM phi(T) = N_e^2 dV/dT (Mason 1994 Fig. 3).

    Args:
        log_t: log10 of the electron temperature in K.
        log_t_peak: log10 of the DEM peak temperature.
        sigma: Width of the bell shape in log10 T units.
        amplitude: Peak DEM in cm^-5 K^-1 (illustrative).

    Returns:
        DEM evaluated at each log_t.
    """
    z = (log_t - log_t_peak) / sigma
    return amplitude * np.exp(-0.5 * z * z)


# Five representative SOHO/CDS-era VUV lines spanning 1e5 - 3e6 K.
LINES: tuple[CoronalLine, ...] = (
    CoronalLine("O V 1218 A", log_t_peak=5.4, sigma_log_t=0.15,
                g_peak=2.5e-23, abundance=4.9e-4),
    CoronalLine("Mg X 625 A", log_t_peak=6.05, sigma_log_t=0.15,
                g_peak=8.0e-24, abundance=4.0e-5),
    CoronalLine("Si XII 521 A", log_t_peak=6.3, sigma_log_t=0.15,
                g_peak=5.0e-24, abundance=3.6e-5),
    CoronalLine("Fe XII 195 A", log_t_peak=6.2, sigma_log_t=0.15,
                g_peak=1.8e-23, abundance=3.2e-5),
    CoronalLine("Fe XV 284 A", log_t_peak=6.35, sigma_log_t=0.15,
                g_peak=1.2e-23, abundance=3.2e-5),
)

In [ ]:
log_t_grid = np.linspace(4.5, 7.0, 500)
t_grid = 10.0**log_t_grid
dem = quiet_sun_dem(log_t_grid)

# Synthesize the integral I = (N(X)/N(H)) * integral G(T) * phi(T) dT.
# We integrate in T (not log T), so dT = T * ln(10) * d(log T).
intensities: dict[str, float] = {}
for line in LINES:
    g = line.g_of_t(log_t_grid)
    integrand = g * dem * t_grid * np.log(10.0)
    intensity = line.abundance * np.trapezoid(integrand, log_t_grid)
    intensities[line.label] = intensity

# Visualisation: DEM with G(T) overlays.
fig, (ax_dem, ax_g) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax_dem.semilogy(log_t_grid, dem, lw=2, color="k",
                label=r"$\varphi(T) = N_e^2\,dV/dT$ (quiet Sun)")
ax_dem.set_ylabel(r"DEM $\varphi(T)$")
ax_dem.set_title("Mason 1994 Fig. 3 -- idealised quiet-Sun DEM and line G(T)")
ax_dem.legend()

for line in LINES:
    ax_g.semilogy(log_t_grid, line.g_of_t(log_t_grid), lw=1.6, label=line.label)
ax_g.set_ylim(1e-26, 1e-22)
ax_g.set_xlabel(r"$\log_{10} T_e\ \mathrm{[K]}$")
ax_g.set_ylabel(r"Contribution function $G(T)$")
ax_g.legend(fontsize=9, ncol=2)
fig.tight_layout()
plt.show()

print("\nSynthesised line intensities (illustrative units):")
for label, value in intensities.items():
    print(f"  {label:14s}  I = {value:8.3e}")

## Part 4: Pottasch isothermal EM (Eq. 31) / Pottasch 등온 방출척도

Pottasch (1964) approximated the integral $\int G(T)\varphi(T)\,dT$ by replacing $G(T)$ with a top-hat of constant height $\bar G$ and width $\Delta\log T$ centred at the line's $T_\mathrm{max}$:

$$\langle\mathrm{EM}\rangle \;=\; \frac{I_\mathrm{obs}}{(N(\mathrm{X})/N(\mathrm{H}))\;\bar G\;T_\mathrm{max}\,\ln(10)\,\Delta\log T}.$$

Pottasch (1964)는 $G(T)$를 $T_\mathrm{max}$ 부근의 직사각형으로 근사하여 위와 같은 등온 평균 EM을 얻었다.

We invert each synthesised intensity from Part 3 with this approximation and verify that the recovered $\langle\mathrm{EM}\rangle$ traces the input DEM $\varphi(T)$.

Part 3에서 합성한 강도를 등온 근사로 역산하여, 회복된 $\langle\mathrm{EM}\rangle$이 입력 DEM의 형태를 재현하는지 확인한다.

In [ ]:
def pottasch_em(intensity: float,
                line: CoronalLine,
                delta_log_t: float = 0.30) -> float:
    """Pottasch isothermal EM approximation (Mason 1994 Eq. 31).

    The line is assumed to form within +/- delta_log_t / 2 of T_max; the
    contribution function is approximated by its peak value times the width.

    Args:
        intensity: Observed (or synthesised) line intensity.
        line: CoronalLine instance (provides g_peak, log_t_peak, abundance).
        delta_log_t: Effective width of G(T) in log10 T units.

    Returns:
        Estimated isothermal column EM in cm^-5 (illustrative units).
    """
    t_max = 10.0**line.log_t_peak
    denom = (line.abundance * line.g_peak * t_max
             * np.log(10.0) * delta_log_t)
    return intensity / denom


recovered_em: list[tuple[float, float]] = []
for line in LINES:
    em_value = pottasch_em(intensities[line.label], line)
    recovered_em.append((line.log_t_peak, em_value))
    print(f"{line.label:14s}  log T_max = {line.log_t_peak:.2f}  "
          f"<EM> = {em_value:.3e}")

# Plot input DEM and Pottasch-recovered points on a comparable axis.
log_t_pts = np.array([x for x, _ in recovered_em])
em_pts = np.array([y for _, y in recovered_em])

# Convert phi(T) [per K] to a column-EM-equivalent for visualisation:
# integrate phi(T) * T * ln10 over a delta_log_t bin (matches Pottasch defn).
delta_log_t = 0.30
phi_columns = (
    quiet_sun_dem(log_t_grid)
    * 10.0**log_t_grid
    * np.log(10.0)
    * delta_log_t
)

fig, ax = plt.subplots()
ax.semilogy(log_t_grid, phi_columns, lw=2, color="k",
            label=r"input $\varphi(T)\,T\,\ln 10\,\Delta\log T$")
ax.scatter(log_t_pts, em_pts, s=120, color="tab:red", zorder=5,
           label="Pottasch isothermal <EM>")
for (lt, em), line in zip(recovered_em, LINES):
    ax.annotate(line.label, (lt, em),
                textcoords="offset points", xytext=(8, -4), fontsize=9)
ax.set_xlabel(r"$\log_{10} T\ \mathrm{[K]}$")
ax.set_ylabel("Column EM (illustrative units)")
ax.set_title("Pottasch (1964) inversion vs input DEM")
ax.legend()
fig.tight_layout()
plt.show()

peak_idx = int(np.argmax(em_pts))
print(f"\nRecovered EM peaks at log T = {log_t_pts[peak_idx]:.2f} "
      f"(input DEM peak at log T = 6.00).")

**Reading / 해석**: The Pottasch points cluster around the input DEM curve and peak near $\log T \approx 6.0$–6.05, demonstrating that even a one-temperature-per-line approximation recovers the gross DEM shape (Mason 1994 Eq. 31). Discrepancies between points reflect the *finite width* of $G(T)$ that Pottasch's top-hat assumption ignores — exactly the systematic error that motivates the full DEM inversion of Eq. 32 (Harrison & Thompson 1992 benchmark).

Pottasch 점들이 입력 DEM 곡선 부근에 모여 있고 $\log T \approx 6.0$–6.05에서 최대값을 가진다. 점들 간의 분산은 $G(T)$의 유한한 폭을 무시한 등온 가정의 한계를 보여준다 — 이것이 Eq. 32 형태의 본격적 DEM 역산이 필요한 이유다.

## Part 5: FIP-effect visualisation (§3.7, Fig. 4) / FIP 효과 시각화

The First-Ionization-Potential (FIP) effect causes coronal abundances of low-FIP elements (Mg, Si, Fe; FIP < 10 eV) to be enhanced by ×3–4 over photospheric values, while high-FIP elements (H, He, C, N, O, Ne, Ar) keep their photospheric abundances. The enhancement ratio depends on the magnetic feature.

FIP 효과: 저-FIP 원소(Mg, Si, Fe; FIP < 10 eV)는 코로나에서 광구보다 ×3–4 *증가*하고, 고-FIP 원소(H, He, C, N, O, Ne, Ar)는 광구 값과 같다. 증가율은 자기 구조에 따라 다르다.

We plot the **[Ne/Mg]** abundance ratio (relative to the photospheric value) for the nine feature types of Mason 1994 Fig. 4 — the spread of >30× across feature types is the central message.

In [ ]:
# Approximate [Ne/Mg] / [Ne/Mg]_photosphere read from Mason 1994 Fig. 4.
# Values are illustrative (within the range stated in section 3.7) and span
# the >30x variation highlighted in the review.
FEATURES: tuple[tuple[str, float], ...] = (
    ("Photosphere", 1.0),
    ("Corona (avg)", 0.25),
    ("Polar plume", 0.10),
    ("Active region", 0.20),
    ("Transition zone", 0.50),
    ("Eruptive prominence", 1.20),
    ("Impulsive flare", 3.00),
    ("SEP (event)", 0.40),
    ("Solar wind", 0.30),
)

names = [n for n, _ in FEATURES]
ratios = np.array([r for _, r in FEATURES])
spread = ratios.max() / ratios.min()

fig, ax = plt.subplots(figsize=(11, 6))
colours = ["tab:gray" if name == "Photosphere" else "tab:purple"
           for name in names]
bars = ax.bar(names, ratios, color=colours, edgecolor="k")
ax.set_yscale("log")
ax.axhline(1.0, color="k", ls=":", alpha=0.7,
           label="photospheric reference")
for bar, value in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width() / 2.0, value * 1.08,
            f"{value:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel(r"[Ne/Mg] / [Ne/Mg]$_\mathrm{photosphere}$")
ax.set_title(
    f"Mason 1994 Fig. 4 -- FIP effect across solar features (spread ~{spread:.0f}x)"
)
ax.tick_params(axis="x", rotation=30)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

print(f"Min [Ne/Mg] = {ratios.min():.2f} (Polar plume)")
print(f"Max [Ne/Mg] = {ratios.max():.2f} (Impulsive flare)")
print(f"Spread max/min = {spread:.1f}x  -- consistent with Mason 1994 'over 30x'.")

**Reading / 해석**: Polar plumes (open field, fast solar wind source) show the strongest FIP fractionation (low [Ne/Mg]); impulsive flares show inverse-FIP enhancement, raising the ratio. The two-decade spread is exactly the *systematic* uncertainty that drives the abundance-choice debate in #61 Abbo+ 2025 (their Asplund-2021 vs CHIANTI-FIP-corrected ×$10^{0.5}$ pair).

폴라 플룸(개방 자기장, 고속 태양풍 원천)은 가장 강한 FIP 분별을 보이고, 임펄시브 플레어는 역-FIP를 보인다. 이 ~30배 변동이 #61 Abbo+ 2025의 abundance 선택 논쟁의 근원이다.

## Summary / 요약

Mason 1994는 VUV 진단법 전체를 한 권에 체계화했고, 본 노트북은 그 §3 핵심 식들 (Eq. 23–32) 을 ChiantiPy 없이 재현했다. 아래 표는 각 진단 기법이 현대 후속 논문들에서 어떻게 진화했는지를 요약한다.

Mason 1994 systematised the full VUV diagnostic toolkit in a single review; this notebook reproduces the key Eq. 23–32 of §3 with analytic two-level / multi-level approximations only. The table below maps each diagnostic to its modern descendant.

| Diagnostic / 진단 | This notebook / 본 노트북 | Modern descendant / 현대 후속 |
|---|---|---|
| Density-sensitive ratio (Eq. 23–28) | Two-level metastable model, O IV, C III | **#63/#64 CHIANTI** — full N-level statistical equilibrium for hundreds of ions |
| Temperature-sensitive ratio (Eq. 29) | O V [629/172] exponential | **#59 Del Zanna & Mason 2018** — band-averaged R(T_e) thermometers |
| Forward EM synthesis | Gaussian DEM × Gaussian G(T) | **#61 Abbo+ 2025** — full CHIANTI G(T) × inverted DEM in streamer plasma |
| Pottasch isothermal EM (Eq. 31) | Top-hat $G$ approximation, recover quiet-Sun DEM | **#59** — full ill-posed DEM inversion with regularisation, multi-line constraints |
| FIP effect (§3.7, Fig. 4) | Bar chart across 9 features | **#61** — Asplund 2021 vs FIP-corrected ×$10^{0.5}$ as the systematic abundance pair |

### Connections / 연결

* **#59 Del Zanna & Mason 2018 (LRSP)** — Mason 본인이 24년 후 작성한 직접 후속 종설. CHIANTI v9가 가능케 한 모든 진단 기법을 본 §3과 같은 순서로 재정리.
* **#61 Abbo+ 2025 (A&A)** — 본 종설의 가장 직접적인 *응용*. R(T_e) (Eq. 29 일반화), DEM (Eq. 32), FIP-corrected abundance (§3.7) 모두 사용.
* **#63/#64 CHIANTI 논문** — 본 §2.2 (collision strengths)와 §2.4 (ionization equilibrium)에서 요구한 "통합 원자 데이터베이스"의 직접적 구현체.

### Caveats / 주의사항

* All G(T) and atomic constants here are illustrative Gaussians / order-unity approximations, **not** ChiantiPy outputs. Numerical values should be interpreted *qualitatively* — they reproduce the *behaviour* of Mason 1994 Eq. 23–32 but not the absolute calibration.
* 본 노트북의 모든 G(T)와 원자 상수는 illustrative 가우시안 / 주문형 근사이며 ChiantiPy 출력이 아니다. 수치는 *정성적*으로 해석할 것 — Mason 1994 Eq. 23-32의 *거동*을 재현하지만 절대 캘리브레이션은 재현하지 않는다.